# panama


In [ ]:
import re

from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, BooleanType, StringType, TimestampType,
)

PAIS_ISO3       = "PAN"
PAIS_NOMBRE     = "Panamá"
# Acepta códigos numéricos (095), rangos (064-071), combinados ("005 y 006") y
# códigos ICD-10 especiales con prefijo de letra (U00-12, U00-U85) tal como
# vienen en los CSV oficiales en formato ancho (INEC) de Panamá.
CAUSA_CODIGO_RE = r"^[A-Z0-9][A-Z0-9 .-]*(?:\s+y\s+[A-Z0-9][A-Z0-9 .-]*)?$"
_SENTINELS      = ("Desconocido", "No Especificado")

# ── DIM_ETARIO grid canónico (docs/modelo_dimensional.md §5.2) ────────────────
# (id_etario, grupo_edad_codigo, grupo_edad_nombre, edad_min, edad_max, categoria_etaria)
_DIM_ETARIO_GRID = [
    (1,  "LT1",   "Menores de 1",      0, 0,    "Niñez"),
    (2,  "01-04", "1 a 4",             1, 4,    "Niñez"),
    (3,  "05-09", "5 a 9",             5, 9,    "Niñez"),
    (4,  "10-14", "10 a 14",          10, 14,   "Niñez"),
    (5,  "15-19", "15 a 19",          15, 19,   "Juventud"),
    (6,  "20-24", "20 a 24",          20, 24,   "Juventud"),
    (7,  "25-29", "25 a 29",          25, 29,   "Juventud"),
    (8,  "30-34", "30 a 34",          30, 34,   "Adultez"),
    (9,  "35-39", "35 a 39",          35, 39,   "Adultez"),
    (10, "40-44", "40 a 44",          40, 44,   "Adultez"),
    (11, "45-49", "45 a 49",          45, 49,   "Adultez"),
    (12, "50-54", "50 a 54",          50, 54,   "Adultez"),
    (13, "55-59", "55 a 59",          55, 59,   "Adultez"),
    (14, "60-64", "60 a 64",          60, 64,   "Adultez"),
    (15, "65-69", "65 a 69",          65, 69,   "Vejez"),
    (16, "70-74", "70 a 74",          70, 74,   "Vejez"),
    (17, "75-79", "75 a 79",          75, 79,   "Vejez"),
    (18, "80-84", "80 a 84",          80, 84,   "Vejez"),
    (19, "85+",   "85 y más",         85, None, "Vejez"),
    (98, "UNK",   "No especificada",  None, None, "No especificado"),
    (99, "ALL",   "Todas las edades", 0, None, "Total"),
]
# id_etario -> categoria_etaria (para roll-up Niñez/Juventud/Adultez/Vejez/Total)
_ETARIO_CAT_MAP = F.create_map(
    *[x for row in _DIM_ETARIO_GRID for x in (F.lit(row[0]), F.lit(row[5]))]
)
# grupo_edad (etiqueta normalizada en español) -> id_etario. Panamá ya colapsa
# 1,2,3,4 -> "1 a 4" en _panama_longify (§5.4), así que el mapeo es directo.
_PAN_ETARIO_MAP = F.create_map(
    *[x for k, v in {
        "Menores de 1": 1, "1 a 4": 2, "5 a 9": 3, "10 a 14": 4,
        "15 a 19": 5, "20 a 24": 6, "25 a 29": 7, "30 a 34": 8,
        "35 a 39": 9, "40 a 44": 10, "45 a 49": 11, "50 a 54": 12,
        "55 a 59": 13, "60 a 64": 14, "65 a 69": 15, "70 a 74": 16,
        "75 a 79": 17, "80 a 84": 18, "85 y más": 19,
        "No especificada": 98,
    }.items() for x in (F.lit(k), F.lit(v))]
)

# ── Normalización de causa a GBD Nivel 2 (docs/modelo_dimensional.md §6) ───────
# Panamá codifica con la Lista abreviada de 103 (INEC). Cada código se cruza a
# CIE-10 (PDF Decodificador) y de ahí a su padre GBD Nivel 2 (Deaths.XLSX).
# Solo se asigna GBD cuando el rango CIE-10 del código colapsa en UNA sola causa
# de las 16; los agregados/residuales ("Resto de…", grupos multi-causa) y los
# capítulos fuera de las 16 (oído 063, piel 082, congénitas 093) quedan en NULL:
# nunca se fuerzan (§6). Generado por scripts/transformation/build_gbd_crosswalk.py
# (data/processed/panama_lista103_gbd.csv).
MAPEO_GBD = {
    "Neoplasms": ("CG0600", 410), "Cardiovascular diseases": ("CG0530", 491),
    "Neglected tropical diseases and malaria": ("CG0250", 344),
    "Nutritional deficiencies": ("CG0470", 386), "Diabetes and kidney diseases": ("CG0510", 955),
    "Mental disorders": ("CG0610", 545), "Neurological disorders": ("CG0590", 542),
    "Self-harm and interpersonal violence": ("CG0860", 717), "COVID-19": ("CG0995", 1013),
    "Transport injuries": ("CG0810", 688), "Digestive diseases": ("CG0580", 526),
    "Unintentional injuries": ("CG0850", 696), "Diarrheal diseases": ("CG0350", 302),
    "HIV/AIDS and sexually transmitted infections": ("CG0290", 366),
    "Respiratory infections and tuberculosis": ("CG0390", 337),
    "Chronic respiratory diseases": ("CG0570", 508),
}
# codigo lista103 -> (cie10, gbd_code, gbd_nombre).
#   (cie10, None, None)         = sin mapear (todo NULL).
#   (cie10, 'CGxxxx', nombre)   = una de las 16 causas de IHME (llave conformada completa).
#   (cie10, None, nombre)       = causa GBD Nivel 2 FUERA de las 16: nombre autoritativo
#       (verificado vs rangos ICD-10 de WHO Mortality) pero gbd_code/cause_id NULL porque
#       el CGxxxx no existe en ninguna fuente confiable (no se fabrica). Ver doc §6.4.
PANAMA_LISTA103 = {
    '001-025': ('A00-B99', None, None),
    '002': ('A00', 'CG0350', 'Diarrheal diseases'),
    '003': ('A09', 'CG0350', 'Diarrheal diseases'),
    '004': ('A01-A08', 'CG0350', 'Diarrheal diseases'),
    '005 y 006': ('A15-A19', 'CG0390', 'Respiratory infections and tuberculosis'),
    '007': ('A20', None, None), '008': ('A33-A35', None, 'Other infectious diseases'), '009': ('A36', None, None),
    '010': ('A37', None, 'Other infectious diseases'), '011': ('A39', None, 'Other infectious diseases'),
    '012': ('A40-A41', None, 'Other infectious diseases'),
    '013': ('A50-A64', 'CG0290', 'HIV/AIDS and sexually transmitted infections'),
    '014': ('A80', None, None),
    '015': ('A82', 'CG0250', 'Neglected tropical diseases and malaria'),
    '016': ('A95', 'CG0250', 'Neglected tropical diseases and malaria'),
    '017': ('A90-A94, A96-A99', 'CG0250', 'Neglected tropical diseases and malaria'),
    '018': ('B05', None, None),
    '019': ('B15-B19', 'CG0580', 'Digestive diseases'),
    '020': ('B20-B24', 'CG0290', 'HIV/AIDS and sexually transmitted infections'),
    '021': ('B50-B54', 'CG0250', 'Neglected tropical diseases and malaria'),
    '022': ('B55', 'CG0250', 'Neglected tropical diseases and malaria'),
    '023': ('B56-B57', 'CG0250', 'Neglected tropical diseases and malaria'),
    '024': ('B65', 'CG0250', 'Neglected tropical diseases and malaria'),
    '025': ('A21-B99 (resto)', None, None),
    '026-046': ('C00-C97', 'CG0600', 'Neoplasms'),
    '027': ('C00-C14', 'CG0600', 'Neoplasms'), '028': ('C15', 'CG0600', 'Neoplasms'),
    '029': ('C16', 'CG0600', 'Neoplasms'), '030': ('C18-C21', 'CG0600', 'Neoplasms'),
    '031': ('C22', 'CG0600', 'Neoplasms'), '032': ('C25', 'CG0600', 'Neoplasms'),
    '033': ('C32', 'CG0600', 'Neoplasms'), '034': ('C33-C34', 'CG0600', 'Neoplasms'),
    '035': ('C43', 'CG0600', 'Neoplasms'), '036': ('C50', 'CG0600', 'Neoplasms'),
    '037': ('C53', 'CG0600', 'Neoplasms'), '038': ('C54-C55', 'CG0600', 'Neoplasms'),
    '039': ('C56', 'CG0600', 'Neoplasms'), '040': ('C61', 'CG0600', 'Neoplasms'),
    '041': ('C67', 'CG0600', 'Neoplasms'), '042': ('C70-C72', 'CG0600', 'Neoplasms'),
    '043': ('C82-C85', 'CG0600', 'Neoplasms'), '044': ('C90', 'CG0600', 'Neoplasms'),
    '045': ('C91-C95', 'CG0600', 'Neoplasms'), '046': ('C17-C97 (resto)', 'CG0600', 'Neoplasms'),
    '047': ('D00-D48', 'CG0600', 'Neoplasms'),
    '048-050': ('D50-D89', None, None), '049': ('D50-D64', None, None), '050': ('D65-D89', None, None),
    '051-054': ('E00-E88', None, None),
    '052': ('E10-E14', 'CG0510', 'Diabetes and kidney diseases'),
    '053': ('E40-E46', 'CG0470', 'Nutritional deficiencies'),
    '054': ('E00-E88 (resto)', None, None),
    '055-057': ('F01-F99', None, None), '056': ('F10-F19', None, 'Substance use disorders'),
    '057': ('F01-F99 (resto)', 'CG0610', 'Mental disorders'),  # resto mental (sin F10-F19 sustancias): capítulo F -> Mental
    '058-061': ('G00-G98', None, None), '059': ('G00, G03', None, 'Other infectious diseases'),
    '060': ('G30', 'CG0590', 'Neurological disorders'),
    '061': ('G04-G98 (resto)', 'CG0590', 'Neurological disorders'),  # resto sistema nervioso (sin meningitis): capítulo G -> Neurological
    '062': ('H00-H59', None, 'Sense organ diseases'), '063': ('H60-H95', None, 'Sense organ diseases'),
    '064-071': ('I00-I99', 'CG0530', 'Cardiovascular diseases'),  # grupo capítulo único (90% Cardiovascular)
    '065': ('I00-I09', 'CG0530', 'Cardiovascular diseases'),
    '066': ('I10-I13', 'CG0530', 'Cardiovascular diseases'),  # hipertensivas: capítulo circulatorio -> Cardiovascular
    '067': ('I20-I25', 'CG0530', 'Cardiovascular diseases'),
    '068': ('I26-I51', 'CG0530', 'Cardiovascular diseases'),
    '069': ('I60-I69', 'CG0530', 'Cardiovascular diseases'),
    '070': ('I70', 'CG0530', 'Cardiovascular diseases'),
    '071': ('I71-I99 (resto)', 'CG0530', 'Cardiovascular diseases'),  # resto circulatorio: capítulo I -> Cardiovascular (igual que 066)
    '072-077': ('J00-J98', None, None),
    '073': ('J10-J11', 'CG0390', 'Respiratory infections and tuberculosis'),
    '074': ('J12-J18', 'CG0390', 'Respiratory infections and tuberculosis'),
    '075': ('J20-J22', 'CG0390', 'Respiratory infections and tuberculosis'),
    '076': ('J40-J47', 'CG0570', 'Chronic respiratory diseases'),
    '077': ('J00-J98 (resto)', None, None),
    '078-081': ('K00-K92', 'CG0580', 'Digestive diseases'),  # grupo capítulo único (94% Digestive)
    '079': ('K25-K27', 'CG0580', 'Digestive diseases'),
    '080': ('K70-K76', 'CG0580', 'Digestive diseases'),
    '081': ('K00-K92 (resto)', 'CG0580', 'Digestive diseases'),  # resto digestivo: capítulo K -> Digestive
    '082': ('L00-L98', None, 'Skin and subcutaneous diseases'), '083': ('M00-M99', None, 'Musculoskeletal disorders'),
    '084-086': ('N00-N98', None, None),
    '085': ('N00-N15', 'CG0510', 'Diabetes and kidney diseases'),  # enf. renales/glomerulares/tubulointersticiales = CKD -> Diabetes and kidney
    '086': ('N17-N98', None, None),
    '087-091': ('O00-O99', None, None), '088': ('O00-O07', None, None), '089': ('O10-O92', None, None),
    '090': ('O98-O99', None, None), '091': ('O95-O97', None, None),
    '092': ('P00-P96', None, 'Maternal and neonatal disorders'),
    '093': ('Q00-Q99', None, 'Congenital birth defects'), '094': ('R00-R99', None, None),
    '095-103': ('V01-Y89', None, None),
    '096': ('V01-V99', 'CG0810', 'Transport injuries'),
    '097': ('W00-W19', 'CG0850', 'Unintentional injuries'),
    '098': ('W65-W74', 'CG0850', 'Unintentional injuries'),
    '099': ('X00-X09', 'CG0850', 'Unintentional injuries'),
    '100': ('X40-X49', 'CG0850', 'Unintentional injuries'),
    '101': ('X60-X84', 'CG0860', 'Self-harm and interpersonal violence'),
    '102': ('X85-Y09', 'CG0860', 'Self-harm and interpersonal violence'),
    '103': ('V01-Y89 (resto) + U00-U85', None, None),
    # ── Alias de los códigos REALES de la data (data/raw/panama) que no calzan
    # con el formato del PDF: años viejos parten Tuberculosis en 005/006, las
    # causas externas vienen solo como grupo 095 (sin 096-103), maternal como
    # 087-090, y los códigos especiales/COVID como U00-12 / U00-U85.
    '005': ('A15-A19', 'CG0390', 'Respiratory infections and tuberculosis'),  # Tuberculosis respiratoria
    '006': ('A15-A19', 'CG0390', 'Respiratory infections and tuberculosis'),  # Otras tuberculosis
    '087-090': ('O00-O99', None, None),   # Embarazo, parto y puerperio (maternal, fuera de las 16)
    '095': ('V01-Y89', None, None),       # Causas externas (grupo; abarca 3 causas GBD -> NULL)
    'U00-12':  ('U00-U12', None, None),   # Códigos especiales (COVID no aislable)
    'U00-U85': ('U00-U85', None, None),   # Códigos especiales (COVID no aislable)
}
_PAN_CIE10_MAP     = F.create_map(*[x for k, (c, gc, gn) in PANAMA_LISTA103.items() for x in (F.lit(k), F.lit(c))])
_PAN_GBDCODE_MAP   = F.create_map(*[x for k, (c, gc, gn) in PANAMA_LISTA103.items() if gc for x in (F.lit(k), F.lit(gc))])
_PAN_GBDNOMBRE_MAP = F.create_map(*[x for k, (c, gc, gn) in PANAMA_LISTA103.items() if gn for x in (F.lit(k), F.lit(gn))])

# codigo lista103 -> nombre de la causa (descripción Lista-103, español). El formato
# ancho no trae el texto de la causa, así que cie10_nombre se puebla desde aquí.
# Tomado de data/processed/panama_lista103_gbd.csv (limpiado de artefactos OCR del PDF)
# + los alias que solo existen en la data real (005, 006, 095, U00-…).
_PAN_NOMBRE = {
    '001-025': "Ciertas enfermedades infecciosas y parasitarias",
    '002': "Cólera",
    '003': "Diarrea y gastroenteritis de presunto origen infeccioso",
    '004': "Otras enfermedades infecciosas intestinales",
    '005 y 006': "Tuberculosis",
    '007': "Peste",
    '008': "Tétanos",
    '009': "Difteria",
    '010': "Tos ferina",
    '011': "Infección meningocócica",
    '012': "Septicemia",
    '013': "Infecciones con un modo de transmisión predominantemente sexual",
    '014': "Poliomielitis aguda",
    '015': "Rabia",
    '016': "Fiebre amarilla",
    '017': "Otras fiebres virales transmitidas por artrópodos y fiebres hemorrágicas virales",
    '018': "Sarampión",
    '019': "Hepatitis viral",
    '020': "Enfermedad por virus de la inmunodeficiencia humana (VIH)",
    '021': "Paludismo (malaria)",
    '022': "Leishmaniasis",
    '023': "Tripanosomiasis",
    '024': "Esquistosomiasis",
    '025': "Las demás enfermedades infecciosas y parasitarias",
    '026-046': "Tumores (neoplasias) malignos",
    '027': "Tumores malignos del labio, de la cavidad bucal y de la faringe",
    '028': "Tumor maligno del esófago",
    '029': "Tumor maligno del estómago",
    '030': "Tumor maligno del colon, del recto y del ano",
    '031': "Tumor maligno del hígado y de las vías biliares intrahepáticas",
    '032': "Tumor maligno del páncreas",
    '033': "Tumor maligno de la laringe",
    '034': "Tumor maligno de la tráquea, de los bronquios y del pulmón",
    '035': "Melanoma maligno de la piel",
    '036': "Tumor maligno de la mama",
    '037': "Tumor maligno del cuello del útero",
    '038': "Tumor maligno de otras partes y de las no especificadas del útero",
    '039': "Tumor maligno del ovario",
    '040': "Tumor maligno de la próstata",
    '041': "Tumor maligno de la vejiga urinaria",
    '042': "Tumor maligno de las meninges, del encéfalo y de otras partes del sistema nervioso central",
    '043': "Linfoma no Hodgkin",
    '044': "Mieloma múltiple y tumores malignos de células plasmáticas",
    '045': "Leucemia",
    '046': "Los demás tumores (neoplasias) malignos",
    '047': "Resto de tumores",
    '048-050': "Enfermedades de la sangre y de los órganos hematopoyéticos, y ciertos trastornos que afectan el mecanismo de la inmunidad",
    '049': "Anemias",
    '050': "Resto de enfermedades de la sangre y de los órganos hematopoyéticos, y ciertos trastornos que afectan el mecanismo de la inmunidad",
    '051-054': "Enfermedades endocrinas, nutricionales y metabólicas",
    '052': "Diabetes mellitus",
    '053': "Desnutrición",
    '054': "Resto de enfermedades endocrinas, nutricionales y metabólicas",
    '055-057': "Trastornos mentales y del comportamiento",
    '056': "Trastornos mentales y del comportamiento debidos al uso de sustancias psicoactivas",
    '057': "Resto de trastornos mentales y del comportamiento",
    '058-061': "Enfermedades del sistema nervioso",
    '059': "Meningitis",
    '060': "Enfermedad de Alzheimer",
    '061': "Resto de enfermedades del sistema nervioso",
    '062': "Enfermedades del ojo y sus anexos",
    '063': "Enfermedades del oído y de la apófisis mastoides",
    '064-071': "Enfermedades del sistema circulatorio",
    '065': "Fiebre reumática aguda y enfermedades cardíacas reumáticas crónicas",
    '066': "Enfermedades hipertensivas",
    '067': "Enfermedades isquémicas del corazón",
    '068': "Otras enfermedades del corazón",
    '069': "Enfermedades cerebrovasculares",
    '070': "Aterosclerosis",
    '071': "Resto de enfermedades del sistema circulatorio",
    '072-077': "Enfermedades del sistema respiratorio",
    '073': "Influenza (gripe)",
    '074': "Neumonía",
    '075': "Otras infecciones agudas de las vías respiratorias inferiores",
    '076': "Enfermedades crónicas de las vías respiratorias inferiores",
    '077': "Resto de enfermedades del sistema respiratorio",
    '078-081': "Enfermedades del sistema digestivo",
    '079': "Úlcera gástrica y duodenal",
    '080': "Enfermedades del hígado",
    '081': "Resto de enfermedades del sistema digestivo",
    '082': "Enfermedades de la piel y del tejido subcutáneo",
    '083': "Enfermedades del sistema osteomuscular y del tejido conjuntivo",
    '084-086': "Enfermedades del sistema genitourinario",
    '085': "Enfermedades renales, glomerulares y tubulointersticiales",
    '086': "Resto de enfermedades del sistema genitourinario",
    '087-091': "Embarazo, parto y puerperio",
    '088': "Embarazo terminado en aborto",
    '089': "Causas obstétricas directas",
    '090': "Causas obstétricas indirectas",
    '091': "Resto de embarazo, parto y puerperio",
    '092': "Ciertas afecciones originadas en el período perinatal",
    '093': "Malformaciones congénitas, deformidades y anomalías cromosómicas",
    '094': "Síntomas y signos no clasificados en otra parte",
    '095-103': "Causas externas de mortalidad",
    '096': "Accidentes de transporte",
    '097': "Caídas",
    '098': "Ahogamiento y sumersión accidentales",
    '099': "Exposición al humo, fuego y llamas",
    '100': "Envenenamiento accidental por, y exposición a sustancias nocivas",
    '101': "Lesiones autoinfligidas intencionalmente",
    '102': "Agresiones",
    '103': "Las demás causas externas",
    '005': "Tuberculosis respiratoria",
    '006': "Otras tuberculosis",
    '087-090': "Embarazo, parto y puerperio",
    '095': "Causas externas de morbilidad y mortalidad",
    'U00-12': "Códigos para propósitos especiales",
    'U00-U85': "Códigos para propósitos especiales",
}
_PAN_NOMBRE_MAP = F.create_map(*[x for k, v in _PAN_NOMBRE.items() for x in (F.lit(k), F.lit(v))])
# codigo lista103 -> gbd_cause_id (vía gbd_nombre -> MAPEO_GBD), para conformar
# DIM_CAUSA igual que ihme/who (§6.4). Solo donde hay GBD; el resto queda NULL.
_PAN_GBDCAUSEID_MAP = F.create_map(
    *[x for k, (c, gc, gn) in PANAMA_LISTA103.items() if gn and gn in MAPEO_GBD
        for x in (F.lit(k), F.lit(MAPEO_GBD[gn][1]))]
)

# ── Alcance (ver docs/modelo_dimensional.md §2.5 y §7 #3) ───────────────────
# FACT_PANAMA queda reducido a una sola sub-tabla: causa_edad_sexo
# (edad/sexo/causa/país/año/defunciones). Se descartan provincia, certificación,
# estado conyugal, actividad económica y tasa. La data local solo trae
# causa_edad_sexo, así que esto coincide con lo que realmente existe en raw.

# Esquema final de FACT_PANAMA, alineado con las columnas comunes de las otras
# tablas stage (ihme/ine/who): `id`, `pais`+`pais_iso3`, booleanos de sexo,
# `causa`, el bloque CIE-10/GBD (cie10_code/cie10_nombre/gbd_code/gbd_nombre/
# gbd_cause_id) y la auditoría (source_file/record_hash/ingestion_ts).
# Específicos de Panamá: `causa_codigo` (Lista-103), `defunciones` (la medida) y
# `nivel`.
# `nivel` (grupo|simple): Panamá trae subtotales por causa (el código de grupo
#   064-071 = suma de sus simples 065-071). Se conservan AMBOS para no perder las
#   defunciones que solo existen como grupo (p.ej. externas 095); el doble conteo
#   se evita filtrando un solo `nivel` al sumar (regla §4, capa hecho/BI).
SCHEMA_PANAMA = {
    "id":           LongType(),
    "pais":         StringType(),
    "pais_iso3":    StringType(),
    "sexo":         StringType(),
    "es_masculino": BooleanType(),
    "es_femenino":  BooleanType(),
    "es_total":     BooleanType(),
    "es_desconocido": BooleanType(),
    "grupo_edad":   StringType(),
    "id_etario":    IntegerType(),
    "categoria_etaria": StringType(),
    "causa_codigo": StringType(),
    "causa":        StringType(),
    "cie10_code":   StringType(),
    "cie10_nombre": StringType(),
    "gbd_code":     StringType(),
    "gbd_nombre":   StringType(),
    "gbd_cause_id": IntegerType(),
    "nivel":        StringType(),
    "anio":         IntegerType(),
    "defunciones":  LongType(),
    "source_file":  StringType(),
    "record_hash":  StringType(),
    "ingestion_ts": TimestampType(),
}

# ── Helpers comunes ────────────────────────────────────────────────────────

def _verify_schema(df: DataFrame, expected: dict, table_name: str) -> None:
    actual = {f.name: f.dataType for f in df.schema.fields}
    errors = []
    for col, exp_type in expected.items():
        if col not in actual:
            errors.append(f"  MISSING  '{col}'")
        elif type(actual[col]) != type(exp_type):
            errors.append(f"  MISMATCH '{col}': expected {exp_type}, got {actual[col]}")
    extra = [c for c in actual if c not in expected]
    if extra:
        errors.append(f"  EXTRA    {extra}")
    if errors:
        print(f"[SCHEMA] {table_name} — {len(errors)} problema(s):")
        for msg in errors:
            print(msg)
    else:
        print(f"[SCHEMA] {table_name} — OK ({len(actual)} columnas, tipos correctos)")


def _replace_invalid_markers(df: DataFrame) -> DataFrame:
    return (
        df
        .replace("-",        None)
        .replace("..",       None)
        .replace("Ignorado", None)
        .replace("ignorado", None)
        .replace("IGNORADO", None)
    )


def _validate_sexo(df: DataFrame) -> DataFrame:
    return df.withColumn("sexo",
        F.when(F.col("sexo").isin("Total", "Hombres", "Mujeres"), F.col("sexo"))
         .otherwise(F.lit(None).cast(StringType()))
    )


def _add_sex_booleans(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("_sx", F.lower(F.trim(F.col("sexo"))))
        .withColumn("es_masculino",
            F.when(F.col("_sx") == "hombres", F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_femenino",
            F.when(F.col("_sx") == "mujeres", F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_total",
            F.when(F.col("_sx") == "total",   F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_desconocido",
            F.when(F.col("_sx").isin("hombres", "mujeres", "total"), F.lit(False))
             .otherwise(F.lit(True)))
        .drop("_sx")
    )


def _normalize_causa(df: DataFrame) -> DataFrame:
    # Los CSV tienen saltos de línea y espacios múltiples embebidos en el texto.
    return df.withColumn("causa",
        F.when(F.col("causa").isNotNull(),
               F.trim(F.regexp_replace(F.col("causa"), r"[\s]+", " ")))
         .otherwise(F.lit(None).cast(StringType()))
    )


def _validate_causa_codigo(df: DataFrame) -> DataFrame:
    return df.withColumn("causa_codigo",
        F.when(
            F.col("causa_codigo").isNotNull()
            & F.trim(F.col("causa_codigo")).rlike(CAUSA_CODIGO_RE),
            F.trim(F.col("causa_codigo"))
        ).otherwise(F.lit(None).cast(StringType()))
    )


def _cast_year(df: DataFrame) -> DataFrame:
    as_int = F.col("anio").cast(DoubleType()).cast(IntegerType())
    return df.withColumn("anio",
        F.when(as_int.between(1900, 2030), as_int)
         .otherwise(F.lit(None).cast(IntegerType()))
    )


def _cast_count(df: DataFrame, col: str) -> DataFrame:
    as_double = F.col(col).cast(DoubleType())
    return df.withColumn(col,
        F.when(as_double >= 0, as_double.cast(LongType()))
         .otherwise(F.lit(None).cast(LongType()))
    )


def _apply_sentinels(df: DataFrame, cols: list) -> DataFrame:
    for c in cols:
        if c in df.columns:
            df = df.withColumn(c, F.trim(F.col(c)))
            df = df.withColumn(c,
                F.when(F.col(c).isin(*_SENTINELS), F.lit(None).cast(StringType()))
                 .otherwise(F.col(c))
            )
    return df


# ── Lectura del formato ancho INEC ──────────────────────────────────────────
# Los CSV oficiales (panama_AAAA_csv.csv, separador ';') vienen en formato ancho:
#   Código; Sexo; Menores de 1; 1; 2; ...; 85 y más; No especificada
# Una fila por (código lista103, sexo) y una columna por grupo de edad. Hay que
# normalizar encabezados, despivotar a largo y derivar el año del nombre de archivo
# antes de aplicar la limpieza/crosswalk comunes.

def _normalize_panama_headers(df: DataFrame) -> DataFrame:
    # Mismo saneo que aplica el sandbox al guardar el raw (Delta no admite
    # ' ,;{}()=\t\n' en nombres de columna): caracteres inválidos -> '_'. Es
    # idempotente, así que funciona tanto si el raw llega saneado como crudo.
    return df.toDF(*[re.sub(r"[ ,;{}()=\t\n]+", "_", c) for c in df.columns])


def _panama_age_columns(df: DataFrame) -> list[str]:
    return [
        c for c in df.columns
        if c not in {"Código", "Sexo", "_source_file", "_ingestion_timestamp"}
    ]


# ── Normalización de grupos de edad (docs/modelo_dimensional.md §4, §5.3-5.4) ──
# Los CSV traen bandas inconsistentes entre años: 2015 (años simples 1-4 + el
# subtotal solapado "Menores de 5"), 2016-19 ("1 a 4"), 2020-24 (años simples).
# Regla del modelo: colapsar 1,2,3,4 -> "1 a 4" (sumando) para que la banda 1-4
# signifique lo mismo en los 10 años, y DESCARTAR "Menores de 5" (solapa LT1+1-4).
# Toda etiqueta que mapee a None se descarta.
# Llave = nombre de columna saneado que llega del raw (espacios -> '_');
# valor = banda canónica legible que va al output `grupo_edad`.
_PAN_EDAD_NORM = {
    "Menores_de_1": "Menores de 1",
    "1": "1 a 4", "2": "1 a 4", "3": "1 a 4", "4": "1 a 4", "1_a_4": "1 a 4",
    "Menores_de_5": None,   # subtotal solapado -> descartar (§5.4 #1)
    "5_a_9": "5 a 9", "10_a_14": "10 a 14", "15_a_19": "15 a 19",
    "20_a_24": "20 a 24", "25_a_29": "25 a 29", "30_a_34": "30 a 34",
    "35_a_39": "35 a 39", "40_a_44": "40 a 44", "45_a_49": "45 a 49",
    "50_a_54": "50 a 54", "55_a_59": "55 a 59", "60_a_64": "60 a 64",
    "65_a_69": "65 a 69", "70_a_74": "70 a 74", "75_a_79": "75 a 79",
    "80_a_84": "80 a 84", "85_y_más": "85 y más",
    "No_especificada": "No especificada",
}
_PAN_EDAD_MAP = F.create_map(
    *[x for k, v in _PAN_EDAD_NORM.items() if v for x in (F.lit(k), F.lit(v))]
)


def _panama_longify(df: DataFrame) -> DataFrame:
    # Despivota las columnas de grupo de edad a filas (grupo_edad, defunciones) y
    # deriva anio del nombre de archivo. La data ancha no trae nombre de causa, así
    # que `causa` queda NULL (cie10_nombre se poblará desde aquí). Solo se conservan
    # filas Hombres/Mujeres (la data no incluye Total). Tras despivotar se normaliza
    # la banda etaria y se SUMA por banda canónica: esto colapsa 1,2,3,4 -> "1 a 4",
    # descarta "Menores de 5" (mapea a None) y absorbe los NULL que deja el union de
    # esquemas heterogéneos entre años (cada año solo llena las columnas que tiene).
    age_columns = _panama_age_columns(df)
    age_structs = [
        F.struct(
            F.lit(col).alias("grupo_edad"),
            F.when(
                F.col(f"`{col}`").isNull()
                | (F.trim(F.col(f"`{col}`")) == "")
                | F.col(f"`{col}`").isin("-", "..", "Ignorado", "ignorado", "IGNORADO"),
                F.lit(None).cast(DoubleType()),
            ).otherwise(F.col(f"`{col}`").cast(DoubleType())).alias("defunciones"),
        )
        for col in age_columns
    ]

    long = (
        df
        .filter(F.col("Sexo").isin("Hombres", "Mujeres"))
        .select(
            F.col("Código").alias("causa_codigo"),
            F.col("Sexo").alias("sexo"),
            F.col("_source_file"),
            F.col("_ingestion_timestamp"),
            F.explode(F.array(*age_structs)).alias("age"),
        )
        .withColumn(
            "anio",
            F.regexp_extract(F.col("_source_file"), r"panama_(\d{4})_csv\.csv", 1)
             .cast(IntegerType()),
        )
        .withColumn("grupo_edad", _PAN_EDAD_MAP[F.col("age.grupo_edad")])
        .filter(F.col("grupo_edad").isNotNull())   # descarta 'Menores de 5'/no mapeadas
    )

    return (
        long
        .groupBy("causa_codigo", "sexo", "anio", "grupo_edad", "_source_file")
        .agg(
            F.sum(F.col("age.defunciones")).cast(LongType()).alias("defunciones"),
            F.max("_ingestion_timestamp").alias("_ingestion_timestamp"),
        )
        .select(
            F.lit(PAIS_ISO3).alias("pais_iso3"),
            F.col("anio"),
            F.col("causa_codigo"),
            F.lit(None).cast(StringType()).alias("causa"),
            F.col("sexo"),
            F.col("grupo_edad"),
            F.col("defunciones"),
            F.col("_source_file"),
            F.col("_ingestion_timestamp"),
        )
    )


def _base_clean(df: DataFrame) -> DataFrame:
    """Limpieza común al hecho de Panamá."""
    df = _replace_invalid_markers(df)
    df = _validate_sexo(df)
    df = _cast_year(df)
    df = _add_sex_booleans(df)
    df = df.dropDuplicates()
    df = (
        df
        .withColumn("pais",      F.lit(PAIS_NOMBRE))
        .withColumn("pais_iso3", F.lit(PAIS_ISO3))
        .withColumn("id", F.monotonically_increasing_id().cast(LongType()))
        .withColumnRenamed("_source_file",         "source_file")
        .withColumnRenamed("_ingestion_timestamp", "ingestion_ts")
    )
    return df


# ── Transform ──────────────────────────────────────────────────────────────

def _add_gbd_columns(df: DataFrame) -> DataFrame:
    # Cruza el código lista103 (causa_codigo crudo, trim) a CIE-10 + GBD Nivel 2.
    # `causa` y `cie10_nombre` = descripción de la causa (Lista-103); el formato
    # ancho no trae texto, así que ambos se pueblan desde _PAN_NOMBRE_MAP (la única
    # etiqueta en español disponible para ese rango CIE-10). cie10_code = rango del PDF.
    cod    = F.trim(F.col("causa_codigo"))
    nombre = _PAN_NOMBRE_MAP[cod]
    df = (
        df
        .withColumn("causa",        nombre)
        .withColumn("cie10_code",   _PAN_CIE10_MAP[cod])
        .withColumn("cie10_nombre", nombre)
        .withColumn("gbd_code",     _PAN_GBDCODE_MAP[cod])
        .withColumn("gbd_nombre",   _PAN_GBDNOMBRE_MAP[cod])
        .withColumn("gbd_cause_id", _PAN_GBDCAUSEID_MAP[cod].cast(IntegerType()))
    )
    # Limpiar cie10_code: solo conservar códigos CIE-10 individuales válidos
    # (A00-Z99 con subcódigo opcional). Los rangos ("C82-C85"), listas
    # ("G00, G03"), y códigos con "resto" que vienen del PDF → NULL.
    # El GBD code sigue intacto; cie10_code es best-effort (§6.3).
    CIE10_RE = r"^[A-Z][0-9]{2}[0-9A-Z]?$"
    return df.withColumn("cie10_code",
        F.when(
            F.col("cie10_code").isNotNull()
            & F.upper(F.trim(F.col("cie10_code"))).rlike(CIE10_RE),
            F.upper(F.trim(F.col("cie10_code")))
        ).otherwise(F.lit(None).cast(StringType()))
    )


_GROUP_CODE_RE = r"^\d{3}-\d{3}$"


def _add_nivel(df: DataFrame) -> DataFrame:
    # 'grupo' = código agregado (rango NNN-NNN, externas 095, especiales U…); el
    # resto es 'simple'. Permite filtrar un solo nivel y evitar el doble conteo.
    cod = F.trim(F.col("causa_codigo"))
    es_grupo = cod.rlike(_GROUP_CODE_RE) | cod.isin("095", "U00-12", "U00-U85")
    return df.withColumn("nivel", F.when(es_grupo, F.lit("grupo")).otherwise(F.lit("simple")))


def _add_record_hash(df: DataFrame) -> DataFrame:
    # Linaje por fila (§3.2): hash de la llave natural del grano.
    keys = ["anio", "causa_codigo", "sexo", "grupo_edad"]
    vals = [F.coalesce(F.col(k).cast(StringType()), F.lit("∅")) for k in keys]
    return df.withColumn("record_hash", F.sha2(F.concat_ws("||", *vals), 256))


def _add_etario_columns(df: DataFrame) -> DataFrame:
    # Panamá ya trae grupo_edad normalizado a etiquetas canónicas (Menores de 1,
    # 1 a 4 … 85 y más, No especificada) vía _panama_longify (colapsa 1-4 y
    # descarta Menores de 5). Mapeo directo etiqueta -> id_etario (§5.3).
    return (
        df
        .withColumn("id_etario", _PAN_ETARIO_MAP[F.col("grupo_edad")].cast(IntegerType()))
        .withColumn("categoria_etaria", _ETARIO_CAT_MAP[F.col("id_etario")])
    )


def transform_panama(df: DataFrame) -> DataFrame:
    df = _normalize_panama_headers(df)
    df = _panama_longify(df)
    df = _base_clean(df)
    df = _normalize_causa(df)
    df = _add_gbd_columns(df)
    df = _validate_causa_codigo(df)
    df = _cast_count(df, "defunciones")
    df = _apply_sentinels(df, ["cie10_nombre", "grupo_edad"])
    df = _add_nivel(df)
    df = _add_record_hash(df)
    df = _add_etario_columns(df)
    return df.select(*SCHEMA_PANAMA.keys())


# ── Ejecución ──────────────────────────────────────────────────────────────

DST = "stage.panama"

spark.sql("CREATE SCHEMA IF NOT EXISTS stage")

try:
    raw    = spark.read.table("semi2.sandbox.raw_panama")
    staged = transform_panama(raw)
    _verify_schema(staged, SCHEMA_PANAMA, DST)
    display(staged)
    # Drop previo: el esquema cambió (12 columnas nuevas); evita conflictos de
    # tipos/columnas con cualquier versión anterior de la tabla.
    spark.sql(f"DROP TABLE IF EXISTS {DST}")
    staged.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(DST)
    print(f"Escrito: {DST} ({staged.count()} filas)")
except Exception as e:
    print(f"Error al procesar sandbox.raw_panama: {e}")
    raise
